In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/hotel_bookings_clean.csv")

conn = sqlite3.connect("../data/hotel.db")
df.to_sql("bookings", conn, if_exists="replace", index=False)

def run_query(query):
    return pd.read_sql(query, conn)

run_query("SELECT COUNT(*) AS total_bookings FROM bookings")

,total_bookings
0,118563


In [2]:
run_query("""SELECT
    hotel,
    COUNT(*)                          AS bookings,
    SUM(is_canceled)                  AS canceled,
    ROUND(AVG(is_canceled) * 100, 1)  AS cancel_rate_pct,
    ROUND(AVG(adr), 2)                AS avg_daily_rate,
    ROUND(SUM(lost_revenue), 0)       AS lost_revenue
FROM bookings
GROUP BY hotel
ORDER BY lost_revenue DESC""")

,hotel,bookings,canceled,cancel_rate_pct,avg_daily_rate,lost_revenue
0,City Hotel,78898,33065,41.9,105.79,10879660.0
1,Resort Hotel,39665,11110,28.0,95.90,5842177.0


In [3]:
run_query("""SELECT
    market_segment,
    COUNT(*)                          AS bookings,
    ROUND(AVG(is_canceled) * 100, 1)  AS cancel_rate_pct,
    ROUND(SUM(lost_revenue), 0)       AS lost_revenue,
    ROUND(SUM(lost_revenue) * 100.0 / SUM(SUM(lost_revenue)) OVER (), 1) AS pct_of_total_loss
FROM bookings
GROUP BY market_segment
HAVING COUNT(*) >= 100
ORDER BY lost_revenue DESC""")

,market_segment,bookings,cancel_rate_pct,lost_revenue,pct_of_total_loss
0,Online TA,56129,36.9,10227646.0,61.2
1,Groups,19758,61.2,2800544.0,16.7
2,Offline TA/TO,24048,34.5,2486958.0,14.9
3,Direct,12449,15.5,993410.0,5.9
4,Corporate,5231,18.9,196383.0,1.2
5,Aviation,231,22.1,16578.0,0.1
6,Complementary,715,12.2,270.0,0.0


In [4]:
run_query("""SELECT
    CASE
        WHEN lead_time <= 7   THEN '1. 0-7 days'
        WHEN lead_time <= 30  THEN '2. 8-30 days'
        WHEN lead_time <= 90  THEN '3. 31-90 days'
        WHEN lead_time <= 180 THEN '4. 91-180 days'
        WHEN lead_time <= 365 THEN '5. 181-365 days'
        ELSE '6. 365+ days'
    END                               AS lead_time_bucket,
    COUNT(*)                          AS bookings,
    ROUND(AVG(is_canceled) * 100, 1)  AS cancel_rate_pct,
    ROUND(SUM(lost_revenue), 0)       AS lost_revenue
FROM bookings
GROUP BY lead_time_bucket
ORDER BY lead_time_bucket""")

,lead_time_bucket,bookings,cancel_rate_pct,lost_revenue
0,1. 0-7 days,19246,9.7,449425.0
1,2. 8-30 days,18848,28.0,2045797.0
2,3. 31-90 days,29436,37.8,4321163.0
3,4. 91-180 days,26384,44.8,5130169.0
4,5. 181-365 days,21505,55.5,4419447.0
5,6. 365+ days,3144,67.7,355836.0


In [5]:
run_query("""SELECT
    market_segment,
    COUNT(*)                          AS bookings,
    ROUND(AVG(is_canceled) * 100, 1)  AS cancel_rate_pct,
    ROUND(AVG(lead_time), 0)          AS avg_lead_time
FROM bookings
WHERE deposit_type = 'Non Refund'
GROUP BY market_segment
ORDER BY bookings DESC""")

,market_segment,bookings,cancel_rate_pct,avg_lead_time
0,Groups,9172,99.3,232.0
1,Offline TA/TO,5005,99.9,189.0
2,Corporate,334,93.4,69.0
3,Online TA,56,94.6,97.0
4,Direct,19,84.2,93.0


In [6]:
run_query("""SELECT
    country,
    COUNT(*)                          AS bookings,
    SUM(is_canceled)                  AS canceled,
    ROUND(AVG(is_canceled) * 100, 1)  AS cancel_rate_pct
FROM bookings
GROUP BY country
HAVING COUNT(*) >= 1000
ORDER BY cancel_rate_pct DESC
LIMIT 10""")

,country,bookings,canceled,cancel_rate_pct
0,PRT,47942,27493,57.3
1,BRA,2217,829,37.4
2,ITA,3756,1332,35.5
3,ESP,8544,2176,25.5
4,CHE,1724,428,24.8
5,IRL,3373,832,24.7
6,USA,2089,501,24.0
7,SWE,1019,227,22.3
8,GBR,12094,2451,20.3
9,BEL,2338,473,20.2


In [7]:
run_query("""WITH monthly AS (
    SELECT
        strftime('%Y-%m', arrival_date)   AS year_month,
        COUNT(*)                          AS bookings,
        ROUND(AVG(is_canceled) * 100, 1)  AS cancel_rate_pct,
        ROUND(SUM(lost_revenue), 0)       AS lost_revenue
    FROM bookings
    GROUP BY year_month
)
SELECT
    year_month,
    bookings,
    cancel_rate_pct,
    lost_revenue,
    ROUND(cancel_rate_pct - LAG(cancel_rate_pct) OVER (ORDER BY year_month), 1) AS rate_change_vs_prev_month,
    SUM(lost_revenue) OVER (ORDER BY year_month)                                AS cumulative_lost_revenue
FROM monthly
ORDER BY year_month""")

,year_month,bookings,cancel_rate_pct,lost_revenue,rate_change_vs_prev_month,cumulative_lost_revenue
0,2015-07,2754,45.7,388522.0,NaN,388522.0
1,2015-08,3863,41.4,560903.0,-4.3,949425.0
2,2015-09,5098,41.1,611535.0,-0.3,1560960.0
3,2015-10,4916,35.2,385491.0,-5.9,1946451.0
4,2015-11,2313,21.0,116280.0,-14.2,2062731.0
5,2015-12,2876,33.8,243827.0,12.8,2306558.0
6,2016-01,2221,24.9,139173.0,-8.9,2445731.0
7,2016-02,3856,34.5,304055.0,9.6,2749786.0
8,2016-03,4781,30.8,454190.0,-3.7,3203976.0
9,2016-04,5397,38.1,626393.0,7.3,3830369.0


In [8]:
run_query("""WITH seg AS (
    SELECT
        hotel,
        market_segment,
        COUNT(*)                          AS bookings,
        ROUND(AVG(is_canceled) * 100, 1)  AS cancel_rate_pct
    FROM bookings
    GROUP BY hotel, market_segment
    HAVING COUNT(*) >= 100
)
SELECT
    hotel,
    market_segment,
    bookings,
    cancel_rate_pct,
    RANK() OVER (PARTITION BY hotel ORDER BY cancel_rate_pct DESC) AS risk_rank
FROM seg
ORDER BY hotel, risk_rank""")

,hotel,market_segment,bookings,cancel_rate_pct,risk_rank
0,City Hotel,Groups,13955,68.9,1
1,City Hotel,Offline TA/TO,16642,43.1,2
2,City Hotel,Online TA,38577,37.5,3
3,City Hotel,Aviation,231,22.1,4
4,City Hotel,Corporate,2961,21.6,5
5,City Hotel,Direct,6016,17.5,6
6,City Hotel,Complementary,514,10.5,7
7,Resort Hotel,Groups,5803,42.6,1
8,Resort Hotel,Online TA,17552,35.6,2
9,Resort Hotel,Complementary,201,16.4,3


In [9]:
run_query("""SELECT
    CASE
        WHEN lead_time > 90 AND total_of_special_requests = 0 AND is_repeated_guest = 0 THEN 'High Risk'
        WHEN lead_time > 30 OR total_of_special_requests = 0 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END                                                AS risk_profile,
    COUNT(*)                                           AS bookings,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_bookings,
    ROUND(AVG(is_canceled) * 100, 1)                   AS cancel_rate_pct,
    ROUND(SUM(lost_revenue), 0)                        AS lost_revenue
FROM bookings
GROUP BY risk_profile
ORDER BY cancel_rate_pct DESC""")

,risk_profile,bookings,pct_of_bookings,cancel_rate_pct,lost_revenue
0,High Risk,32195,27.2,63.9,7029046.0
1,Medium Risk,69316,58.5,30.7,8809564.0
2,Low Risk,17052,14.4,13.7,883227.0


In [10]:
run_query("""SELECT
    CASE WHEN is_canceled = 1 THEN 'Canceled' ELSE 'Not Canceled' END AS status,
    COUNT(*)                     AS bookings,
    ROUND(AVG(adr), 2)           AS avg_daily_rate,
    ROUND(AVG(total_nights), 2)  AS avg_nights,
    ROUND(AVG(lead_time), 0)     AS avg_lead_time
FROM bookings
GROUP BY status""")

,status,bookings,avg_daily_rate,avg_nights,avg_lead_time
0,Canceled,44175,104.96,3.49,145.0
1,Not Canceled,74388,101.01,3.42,80.0


## SQL Key Findings

1. **The deposit paradox is explained:** 97% of non-refundable bookings come from Groups (62.9%) and Offline Travel Agents (34.3%), with an average lead time of 189–232 days. These are block bookings released by agencies — the deposit policy is not the cause of cancellation. *Correlation ≠ causation.*
2. **Portugal drives cancellations:** domestic guests represent ~40% of bookings but ~62% of all cancellations (57.3% cancellation rate vs 20–37% for other countries).
3. **The risk rule works:** High-Risk bookings (lead time > 90 days, no special requests, new guest) are 27% of bookings but cancel at 63.9% — 4.7x the Low-Risk rate — and account for 42% of lost revenue.

## Limitations
- Lost revenue is *expected* revenue from canceled bookings; non-refundable deposits were likely retained, so actual losses are lower.
- Duplicate rows were kept, as the dataset has no booking ID and they likely represent group bookings.
- Prices are in EUR (hotels located in Portugal).